In [82]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time

In [83]:
dataframe1 = cudf.read_csv('household_power_consumption.txt',delimiter = ';')
dataframe1.to_csv('household_power_consumption.csv', index = None)

In [84]:
df = cudf.read_csv('household_power_consumption.csv')
df

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0
...,...,...,...,...,...,...,...,...,...
2075254,26/11/2010,20:58:00,0.946,0.000,240.430,4.000,0.000,0.000,0.0
2075255,26/11/2010,20:59:00,0.944,0.000,240.000,4.000,0.000,0.000,0.0
2075256,26/11/2010,21:00:00,0.938,0.000,239.820,3.800,0.000,0.000,0.0
2075257,26/11/2010,21:01:00,0.934,0.000,239.700,3.800,0.000,0.000,0.0


In [85]:
df.isnull().sum()

Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64

In [86]:
100*(df.isnull().sum())/len(df)

Date                     0.000000
Time                     0.000000
Global_active_power      0.000000
Global_reactive_power    0.000000
Voltage                  0.000000
Global_intensity         0.000000
Sub_metering_1           0.000000
Sub_metering_2           0.000000
Sub_metering_3           1.251844
dtype: float64

In [87]:
df.isnull().sum(axis=0)

Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64

In [88]:
df.dtypes

Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object

In [89]:
df.columns

Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='object')

In [99]:
class nullvalues(object):
    def __init__(self, dataset, char_list):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.df_null_rows = []
        self.index_list = []
        self.dataset_columns = self.dataset.copy().columns
        self.index_list_null = []
        self.index_list_not_null = []
        self.dataset_pandas = self.dataset.to_pandas().reset_index(drop = True)
        self.char_list = char_list
        self.idx = idx = self.dataset_pandas.index

    def nulldataset(self):
        global df_null_global
        global index_list_global
        df_null = self.dataset_pandas.copy()
        
        for index, row in df_null.iterrows():
            if (row.isnull().any() or row.isna().any()):
                self.index_list.append(index)
                index_list_global = self.index_list
        print('nulldataset def is done')
                
    def missingvalues(self):
        global index_list_null_global
        df_null = self.dataset_pandas.copy()
        
        for index, row in df_null.iterrows():
            for j in range(0, len(self.char_list)):
                if (df_null.iloc[index, :] == self.char_list[j]).any():
                    self.index_list_null.append(index)
                    index_list_null_global = self.index_list_null

        print('missingvalues def is done')               
    def mergedIndexLists(self):
        global merged_index_list_global

        merged_index_list = list(set(index_list_global + index_list_null_global))
        merged_index_list_global = merged_index_list

        print('mergedIndexLists def is done')

    def IndexNotNull(self):
        global index_list_not_null_global
        
        self.index_list_not_null = list(set(self.idx) - set(merged_index_list_global))
        index_list_not_null_global = self.index_list_not_null
        print('IndexNotNull def is done')
        
    def DFwithNullValues(self):
        global dfNull_global
        dfNull = self.dataset_pandas.copy()
        dfNull = dfNull.loc[dfNull.copy().index[merged_index_list_global]]
        dfNull_global = dfNull

        print('DFwithNullValues def is done')

    def DFnotNull(self):
        global dfNotNull_global
        dfNotNull_global = self.dataset_pandas.copy()
        for index, row in dfNotNull_global.iterrows():
            for j in range(0, len(index_list_not_null_global)):
                if (index == j):
                    dfNotNull_global.drop(dfNotNull_global.index[[index]] ,inplace=True)
        print('DFnotNull def is done')
                    
    def cudfDataFrame(self):
        global cudf_null_df_global
        global cudf_df_global

        cudf_null_df_global = cudf.DataFrame(dfNull_global)
        cudf_df_global = cudf.DataFrame(dfNotNull_global)

        print('cudfDataFrame def is done')

    def main(self):
        st = time.time()
        self.nulldataset()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.missingvalues()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.mergedIndexLists()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.IndexNotNull()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.DFwithNullValues()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.DFnotNull()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [100]:
char_list = ('?')

In [101]:
null = nullvalues(df, char_list)

In [102]:
null.main()

nulldataset def is done
Execution time: 176.01952242851257 seconds
missingvalues def is done
Execution time: 250.14090013504028 seconds
mergedIndexLists def is done
Execution time: 0.0013275146484375 seconds
IndexNotNull def is done
Execution time: 0.11059904098510742 seconds


In [104]:
len(index_list_not_null_global)

2049280